In [ ]:
import os
import sys
from pathlib import Path
import csv
import zipfile

if "torch" in sys.modules and sys.modules["torch"].cuda.is_initialized():
    raise RuntimeError("CUDA 已初始化。请重启 Kaggle Session，然后从头 Run All。")
already_loaded = [name for name in ("transformers", "peft", "accelerate", "datasets") if name in sys.modules]
if already_loaded:
    raise RuntimeError(f"发现已加载的库 {already_loaded}。请重启 Session，再从头 Run All。")
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

INPUT_ROOT = Path("/kaggle/input")
candidate_dirs = [
    INPUT_ROOT / "competitions/word2vec-nlp-tutorial",
    INPUT_ROOT / "word2vec-nlp-tutorial",
]
if INPUT_ROOT.exists():
    candidate_dirs += [p.parent for p in INPUT_ROOT.rglob("sampleSubmission.csv")]

def locate_inputs(directories):
    for directory in dict.fromkeys(directories):
        found = {}
        for key, stem in (
            ("train", "labeledTrainData.tsv"),
            ("test", "testData.tsv"),
            ("sample", "sampleSubmission.csv"),
        ):
            paths = [directory / stem]
            if stem.endswith(".tsv"):
                paths.append(directory / (stem + ".zip"))
            hit = next((p for p in paths if p.is_file()), None)
            if hit is not None:
                found[key] = hit
        if len(found) == 3:
            return found
    raise FileNotFoundError(
        "未找到比赛数据。请 Add Input 添加 word2vec-nlp-tutorial；"
        "必须同时包含 labeledTrainData.tsv(.zip)、testData.tsv(.zip)、sampleSubmission.csv。"
    )

INPUT_FILES = locate_inputs(candidate_dirs)
for key, path in INPUT_FILES.items():
    print(f"{key}: {path}")

In [ ]:
%pip uninstall --yes torchao
%pip install --quiet "transformers==5.19.0" "peft==0.21.2" "accelerate==1.15.0" "datasets==5.1.0" "protobuf>=4.25,<8"

In [ ]:
# pip uninstall 可能在 Kaggle 上残留 torchao 的 dist-info，PEFT 会读到旧版本并报错，这里彻底清理。
import importlib, importlib.metadata as md, importlib.util, shutil, subprocess

def torchao_metadata_present():
    return any((d.metadata["Name"] or "").lower().replace("_", "-") == "torchao" for d in md.distributions())

for entry in dict.fromkeys(p for p in sys.path if p):
    root = Path(entry)
    if root.is_dir():
        for path in root.glob("torchao*"):
            print("清理残留:", path)
            shutil.rmtree(path, ignore_errors=True) if path.is_dir() else path.unlink(missing_ok=True)
importlib.invalidate_caches()
if importlib.util.find_spec("torchao") is not None or torchao_metadata_present():
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "torchao>=0.16.0"], check=True)
    importlib.invalidate_caches()
    print("torchao 版本:", md.version("torchao"))
else:
    print("torchao 已完全移除。")

In [ ]:
import gc
import html
import json
import math
import re
import time
from datetime import datetime, timezone
from importlib.metadata import version

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from datasets import Dataset
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.model_selection import train_test_split
from transformers import AutoModelForCausalLM, AutoTokenizer, Trainer, TrainerCallback, TrainingArguments, set_seed
from peft import LoraConfig, TaskType, get_peft_model, cast_mixed_precision_params

VERSIONS = {name: version(name) for name in (
    "torch", "transformers", "peft", "accelerate", "datasets", "numpy", "pandas", "scikit-learn",
)}
print(json.dumps(VERSIONS, indent=2))
if not torch.cuda.is_available():
    raise RuntimeError("未检测到 CUDA GPU。请在 Settings 里选 GPU T4 x2 并重启 Session。")
if torch.cuda.device_count() != 1:
    raise RuntimeError("应只看到一张逻辑 GPU。请重启 Session，确认第一段代码最先执行。")
if torch.cuda.get_device_capability(0) < (7, 0):
    raise RuntimeError(f"{torch.cuda.get_device_name(0)} 太旧，请改用 T4。")
print("设备:", torch.cuda.get_device_name(0),
      f"{torch.cuda.get_device_properties(0).total_memory / 2**30:.1f} GiB")

In [ ]:
# SMOKE = True：只用 64 条训练 / 64 条验证 / 64 条测试，几分钟内走通全流程，用来先查错。
SMOKE = False

MODEL_ID = "Qwen/Qwen3-4B-Base"
SEED = 42  # 与 BERT / DeBERTa 实验相同的分层划分，验证 AUC 可以直接对照

# 指令模版与导师脚本相同（Alpaca）。训练时只在 "### Response:\n" 之后的第一个 token 上算损失。
PROMPT = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
Analyze the sentiment of the following movie review. Respond with 'positive' or 'negative'.

### Input:
{}

### Response:
"""
ANSWERS = {0: "negative", 1: "positive"}

# 超长影评保留开头 128 + 结尾 384 个 token（Sun et al. 2019 在 IMDB 上 head+tail 截断效果最好）。
MAX_REVIEW_TOKENS = 512
HEAD_TOKENS = 128

LORA_R, LORA_ALPHA, LORA_DROPOUT = 16, 32, 0.0
LORA_TARGETS = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]

EPOCHS = 1
TRAIN_BATCH, GRAD_ACC = 8, 2  # 有效 batch 16，与导师脚本一致
LEARNING_RATE = 1e-4
WARMUP = 0.05
WEIGHT_DECAY = 0.01
EVAL_TOKENS_PER_BATCH = 16384  # 推理按 token 数凑 batch
ZERO_SHOT_EVAL = True  # 训练前先在验证集上测一次，作为 SFT 前的对照
RESUME_FROM = None  # 中断后续跑：填上一次输出里的 checkpoints/checkpoint-xxx 路径（需先 Add Input 挂载）

RUN_ROOT = Path("/kaggle/working") / (
    ("smoke_" if SMOKE else "") + "imdb_qwen3_4b_sft_" + datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
)
RUN_ROOT.mkdir(parents=True, exist_ok=False)
CONFIG = {k: v for k, v in globals().items() if k.isupper() and isinstance(v, (str, int, float, bool, list, dict))}
CONFIG["RUN_ROOT"] = str(RUN_ROOT)
(RUN_ROOT / "config.json").write_text(json.dumps(CONFIG, indent=2, ensure_ascii=False, default=str) + "\n", encoding="utf-8")
print("输出目录:", RUN_ROOT)

In [ ]:
def save_json(path, value):
    Path(path).write_text(json.dumps(value, indent=2, ensure_ascii=False, default=str) + "\n", encoding="utf-8")

def read_table(path, is_tsv=False):
    kwargs = {"dtype": {"id": str}}
    if is_tsv:
        kwargs.update(sep="\t", quoting=csv.QUOTE_NONE)
    path = Path(path)
    if path.suffix == ".zip":
        expected = path.name[:-4]
        with zipfile.ZipFile(path) as archive:
            members = [n for n in archive.namelist() if Path(n).name == expected and not n.endswith("/")]
            if len(members) != 1:
                raise ValueError(f"{path.name} 内应恰好有一个 {expected}，实际找到 {members}")
            with archive.open(members[0]) as stream:
                return pd.read_csv(stream, **kwargs)
    return pd.read_csv(path, **kwargs)

def normalized_ids(values):
    if values.isna().any():
        raise ValueError("ID 存在缺失值。")
    return values.astype(str).str.strip().str.strip('"').str.strip()

def clean_review(text):
    text = str(text).strip()
    if len(text) >= 2 and text.startswith('"') and text.endswith('"'):
        text = text[1:-1]  # TSV 的外层引号
    text = text.replace('\\"', '"')  # QUOTE_NONE 读入时正文里的引号带反斜杠
    return re.sub(r"\s+", " ", html.unescape(re.sub(r"<[^>]*>", " ", text))).strip()

def validate_frames(train, test, sample, expected_rows=25000):
    for name, frame, required in (
        ("train", train, {"id", "review", "sentiment"}),
        ("test", test, {"id", "review"}),
        ("sample", sample, {"id", "sentiment"}),
    ):
        if not required.issubset(frame.columns):
            raise ValueError(f"{name} 缺少列 {required - set(frame.columns)}")
        if len(frame) != expected_rows:
            raise ValueError(f"{name} 应有 {expected_rows} 条数据，实际 {len(frame)}")
        frame["id"] = normalized_ids(frame["id"])
        if frame["id"].eq("").any() or frame["id"].duplicated().any():
            raise ValueError(f"{name} 的 ID 为空或重复。")
    for name, frame in (("train", train), ("test", test)):
        frame["review"] = frame["review"].map(clean_review)
        if frame["review"].eq("").any():
            raise ValueError(f"{name} 清洗后存在空影评。")
    labels = pd.to_numeric(train["sentiment"], errors="raise")
    if set(labels.unique()) != {0, 1}:
        raise ValueError("训练标签必须是 0/1。")
    train["sentiment"] = labels.astype("int64")
    if set(test["id"]) != set(sample["id"]):
        raise ValueError("测试集与 sampleSubmission 的 ID 集合不一致。")
    return train, test, sample

train_all, test_frame, sample_frame = validate_frames(
    read_table(INPUT_FILES["train"], is_tsv=True),
    read_table(INPUT_FILES["test"], is_tsv=True),
    read_table(INPUT_FILES["sample"]),
)
train_idx, val_idx = train_test_split(
    np.arange(len(train_all)), test_size=0.2, random_state=SEED, stratify=train_all["sentiment"],
)
train_frame = train_all.iloc[train_idx].reset_index(drop=True)
val_frame = train_all.iloc[val_idx].reset_index(drop=True)
if SMOKE:
    train_frame = train_frame.sample(64, random_state=SEED).reset_index(drop=True)
    val_frame = val_frame.sample(64, random_state=SEED).reset_index(drop=True)
    test_frame = test_frame.head(64).copy()
pd.concat([
    train_frame[["id"]].assign(split="train"), val_frame[["id"]].assign(split="validation"),
]).to_csv(RUN_ROOT / "split.csv", index=False)
print(f"训练 {len(train_frame)} / 验证 {len(val_frame)} / 测试 {len(test_frame)}")

In [ ]:
set_seed(SEED)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

ANSWER_IDS = {}
for label, word in ANSWERS.items():
    ids = tokenizer(word, add_special_tokens=False)["input_ids"]
    if len(ids) != 1:
        raise ValueError(f"答案 {word!r} 被切成 {len(ids)} 个 token: {ids}，需要换成单 token 的答案词。")
    ANSWER_IDS[label] = ids[0]
NEG_ID, POS_ID = ANSWER_IDS[0], ANSWER_IDS[1]
print("答案 token:", {ANSWERS[k]: v for k, v in ANSWER_IDS.items()})

def truncate_review(text):
    ids = tokenizer(text, add_special_tokens=False)["input_ids"]
    if len(ids) <= MAX_REVIEW_TOKENS:
        return text, len(ids), False
    head = tokenizer.decode(ids[:HEAD_TOKENS])
    tail = tokenizer.decode(ids[-(MAX_REVIEW_TOKENS - HEAD_TOKENS):])
    return head.rstrip() + " ... " + tail.lstrip(), len(ids), True

def encode_frame(frame, labeled):
    rows = {"input_ids": [], "length": [], "review_tokens": [], "truncated": []}
    if labeled:
        rows["answer_id"] = []
    for i, review in enumerate(frame["review"]):
        text, n_review, cut = truncate_review(review)
        ids = tokenizer(PROMPT.format(text), add_special_tokens=False)["input_ids"]
        rows["input_ids"].append(ids)
        rows["length"].append(len(ids))
        rows["review_tokens"].append(n_review)
        rows["truncated"].append(cut)
        if labeled:
            rows["answer_id"].append(ANSWER_IDS[int(frame["sentiment"].iloc[i])])
    return rows

# 检查分开编码与整体编码一致：提示词 + 答案词一起分词时，前面的 token 不变，最后多出的正好是答案 token。
probe = PROMPT.format(train_frame["review"].iloc[0])
probe_ids = tokenizer(probe, add_special_tokens=False)["input_ids"]
for label, word in ANSWERS.items():
    joint = tokenizer(probe + word, add_special_tokens=False)["input_ids"]
    if joint != probe_ids + [ANSWER_IDS[label]]:
        raise ValueError(f"提示词结尾与答案 {word!r} 分词时发生合并，训练位置会错位。")

t0 = time.time()
train_enc = encode_frame(train_frame, labeled=True)
val_enc = encode_frame(val_frame, labeled=True)
test_enc = encode_frame(test_frame, labeled=False)
for name, enc in (("train", train_enc), ("val", val_enc), ("test", test_enc)):
    lengths = np.array(enc["length"])
    print(f"{name}: 平均 {lengths.mean():.0f} token，最长 {lengths.max()}，"
          f"截断 {np.mean(enc['truncated']) * 100:.1f}%")
print(f"分词用时 {time.time() - t0:.0f} 秒")
print("\n第一条训练样本（解码回文本，末尾之后就是要预测的答案）:\n" + tokenizer.decode(train_enc["input_ids"][0])[-300:])

In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, dtype=torch.float16, device_map={"": 0}, attn_implementation="sdpa",
)
model.config.use_cache = False
MODEL_REVISION = getattr(model.config, "_commit_hash", None)
print("模型版本:", MODEL_REVISION)

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM, r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
    bias="none", target_modules=LORA_TARGETS,
)
model = get_peft_model(model, lora_config)
cast_mixed_precision_params(model, dtype=torch.float16)  # 冻结的底座 fp16，LoRA 参数 fp32
model.print_trainable_parameters()

# 按 LoRA 公式核对可训练参数：每个被替换的线性层增加 r × (输入维 + 输出维)。
cfg = model.base_model.model.config
h, inter = cfg.hidden_size, cfg.intermediate_size
head_dim = getattr(cfg, "head_dim", None) or h // cfg.num_attention_heads
q_out, kv_out = cfg.num_attention_heads * head_dim, cfg.num_key_value_heads * head_dim
dims = {"q_proj": (h, q_out), "k_proj": (h, kv_out), "v_proj": (h, kv_out), "o_proj": (q_out, h),
        "gate_proj": (h, inter), "up_proj": (h, inter), "down_proj": (inter, h)}
expected = cfg.num_hidden_layers * sum(LORA_R * (a + b) for name, (a, b) in dims.items() if name in LORA_TARGETS)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
if trainable != expected:
    raise ValueError(f"可训练参数 {trainable} 与公式 {expected} 不符")
if any(p.dtype != torch.float32 for p in model.parameters() if p.requires_grad):
    raise ValueError("LoRA 参数应为 fp32")
print(f"可训练 {trainable:,} / 总计 {total:,}（{trainable / total * 100:.2f}%），与公式一致")

In [ ]:
def pad_batch(seqs):
    width = max(len(s) for s in seqs)
    ids = torch.full((len(seqs), width), tokenizer.pad_token_id, dtype=torch.long)
    mask = torch.zeros((len(seqs), width), dtype=torch.long)
    for i, s in enumerate(seqs):
        ids[i, :len(s)] = torch.tensor(s, dtype=torch.long)
        mask[i, :len(s)] = 1
    return ids, mask

def last_token_logits(model, input_ids, attention_mask):
    # 右侧补齐，每行最后一个真实 token 的位置 = 有效长度 - 1；只为这些位置算 LM 头，不算整句 15 万维的 logits。
    last = attention_mask.sum(dim=1) - 1
    keep = torch.unique(last)
    out = model(input_ids=input_ids, attention_mask=attention_mask, logits_to_keep=keep, use_cache=False)
    column = torch.searchsorted(keep, last)
    return out.logits[torch.arange(input_ids.size(0), device=input_ids.device), column]

def length_batches(lengths, tokens_per_batch):
    order = np.argsort(-np.asarray(lengths), kind="stable")
    batches, current = [], []
    for i in order:
        if current and lengths[current[0]] * (len(current) + 1) > tokens_per_batch:
            batches.append(current)
            current = []
        current.append(int(i))
    if current:
        batches.append(current)
    return batches

@torch.no_grad()
def score(model, enc, desc):
    model.eval()
    n = len(enc["input_ids"])
    p_pos = np.zeros(n)
    answer_mass = np.zeros(n)
    top_is_answer = np.zeros(n, dtype=bool)
    nll = np.full(n, np.nan)
    batches = length_batches(enc["length"], EVAL_TOKENS_PER_BATCH)
    t0 = time.time()
    for b, idx in enumerate(batches):
        ids, mask = pad_batch([enc["input_ids"][i] for i in idx])
        ids, mask = ids.cuda(), mask.cuda()
        with torch.autocast("cuda", dtype=torch.float16):
            logits = last_token_logits(model, ids, mask)
        logp = torch.log_softmax(logits.float(), dim=-1)
        if not torch.isfinite(logp).all():
            raise FloatingPointError("logits 出现 NaN/Inf")
        lp_pos, lp_neg = logp[:, POS_ID], logp[:, NEG_ID]
        p_pos[idx] = torch.sigmoid(lp_pos - lp_neg).cpu().numpy()  # 只在两个答案词之间归一化
        answer_mass[idx] = (lp_pos.exp() + lp_neg.exp()).cpu().numpy()  # 两个答案词占整个词表概率的多少
        top = logp.argmax(dim=-1)
        top_is_answer[idx] = ((top == POS_ID) | (top == NEG_ID)).cpu().numpy()
        if "answer_id" in enc:
            target = torch.tensor([enc["answer_id"][i] for i in idx], device=logp.device)
            nll[idx] = (-logp.gather(1, target[:, None]).squeeze(1)).cpu().numpy()
        if b == 0 or (b + 1) % 200 == 0 or b + 1 == len(batches):
            done = sum(len(x) for x in batches[:b + 1])
            rate = done / max(time.time() - t0, 1e-9)
            print(f"[{desc}] {done}/{n}  {rate:.1f} 条/秒  预计剩余 {(n - done) / rate / 60:.1f} 分钟", flush=True)
    return {"p_pos": p_pos, "answer_mass": answer_mass, "top_is_answer": top_is_answer, "nll": nll}

def metrics(result, labels):
    labels = np.asarray(labels)
    out = {
        "accuracy": float(accuracy_score(labels, (result["p_pos"] >= 0.5).astype(int))),
        "auc": float(roc_auc_score(labels, result["p_pos"])) if len(set(labels)) == 2 else float("nan"),
        "answer_loss": float(np.nanmean(result["nll"])),
        "mean_answer_mass": float(result["answer_mass"].mean()),
        "top1_is_answer": float(result["top_is_answer"].mean()),
    }
    return out

def save_predictions(frame, result, path):
    out = pd.DataFrame({"id": frame["id"].values, "p_positive": result["p_pos"],
                        "answer_mass": result["answer_mass"], "top1_is_answer": result["top_is_answer"]})
    if "sentiment" in frame:
        out["label"] = frame["sentiment"].values
        out["answer_loss"] = result["nll"]
    out.to_csv(path, index=False)

SUMMARY = {"config": CONFIG, "versions": VERSIONS, "model_revision": MODEL_REVISION,
           "trainable_params": trainable, "total_params": total,
           "gpu": torch.cuda.get_device_name(0)}
if ZERO_SHOT_EVAL:
    t0 = time.time()
    zero = score(model, val_enc, "zero-shot 验证")
    SUMMARY["zero_shot_validation"] = {**metrics(zero, val_frame["sentiment"]), "seconds": round(time.time() - t0)}
    save_predictions(val_frame, zero, RUN_ROOT / "validation_zero_shot.csv")
    save_json(RUN_ROOT / "summary.json", SUMMARY)
    print(json.dumps(SUMMARY["zero_shot_validation"], indent=2))

In [ ]:
class LastTokenTrainer(Trainer):
    # 损失 = 答案 token 在整个词表上的交叉熵，提示词部分不计损失；返回的是 micro batch 平均，由 Trainer 再除以累积步数。
    loss_is_scaled_for_ga = False

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        logits = last_token_logits(model, inputs["input_ids"], inputs["attention_mask"])
        loss = F.cross_entropy(logits.float(), inputs["answer_id"])
        return (loss, logits) if return_outputs else loss

def collate(rows):
    ids, mask = pad_batch([r["input_ids"] for r in rows])
    return {"input_ids": ids, "attention_mask": mask,
            "answer_id": torch.tensor([r["answer_id"] for r in rows], dtype=torch.long)}

class ProgressCallback(TrainerCallback):
    def on_train_begin(self, args, state, control, **kwargs):
        self.t0 = time.time()

    def on_step_end(self, args, state, control, **kwargs):
        step = state.global_step
        if step in (5, 20) or step % 100 == 0 or step == state.max_steps:
            spent = time.time() - self.t0
            print(f"step {step}/{state.max_steps}  {spent / step:.1f} 秒/步  "
                  f"已用 {spent / 60:.1f} 分钟  预计剩余 {(state.max_steps - step) * spent / step / 60:.1f} 分钟",
                  flush=True)

train_dataset = Dataset.from_dict({k: train_enc[k] for k in ("input_ids", "answer_id", "length")})
args = TrainingArguments(
    output_dir=str(RUN_ROOT / "checkpoints"),
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=TRAIN_BATCH,
    gradient_accumulation_steps=GRAD_ACC,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    warmup_steps=WARMUP,
    lr_scheduler_type="linear",
    optim="adamw_torch",
    fp16=True,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    train_sampling_strategy="group_by_length",  # 长度相近的样本放进同一批，少补齐
    length_column_name="length",
    remove_unused_columns=False,
    label_names=["answer_id"],
    eval_strategy="no",
    save_strategy="steps",
    save_steps=250,
    save_total_limit=1,
    logging_steps=10,
    seed=SEED,
    data_seed=SEED,
    dataloader_num_workers=0,
    report_to="none",
)
trainer = LastTokenTrainer(
    model=model, args=args, train_dataset=train_dataset,
    data_collator=collate, processing_class=tokenizer, callbacks=[ProgressCallback()],
)
torch.cuda.reset_peak_memory_stats()
t0 = time.time()
train_result = trainer.train(resume_from_checkpoint=RESUME_FROM)
SUMMARY["train"] = {
    "seconds": round(time.time() - t0),
    "steps": trainer.state.global_step,
    "train_loss": train_result.training_loss,
    "peak_memory_gib": round(torch.cuda.max_memory_allocated() / 2**30, 2),
}
pd.DataFrame(trainer.state.log_history).to_csv(RUN_ROOT / "training_log.csv", index=False)
model.save_pretrained(RUN_ROOT / "adapter")
tokenizer.save_pretrained(RUN_ROOT / "adapter")
save_json(RUN_ROOT / "summary.json", SUMMARY)
print(json.dumps(SUMMARY["train"], indent=2))

In [ ]:
t0 = time.time()
tuned = score(model, val_enc, "SFT 后验证")
SUMMARY["sft_validation"] = {**metrics(tuned, val_frame["sentiment"]), "seconds": round(time.time() - t0)}
save_predictions(val_frame, tuned, RUN_ROOT / "validation_sft.csv")
save_json(RUN_ROOT / "summary.json", SUMMARY)

rows = {"SFT 后": SUMMARY["sft_validation"]}
if "zero_shot_validation" in SUMMARY:
    rows = {"SFT 前（zero-shot）": SUMMARY["zero_shot_validation"], **rows}
print(pd.DataFrame(rows).T.drop(columns="seconds").round(4).to_string())

In [ ]:
t0 = time.time()
test_result = score(model, test_enc, "测试")
SUMMARY["test_seconds"] = round(time.time() - t0)
save_predictions(test_frame, test_result, RUN_ROOT / "test_predictions.csv")

prediction = pd.Series(test_result["p_pos"], index=test_frame["id"].values)
if SMOKE:
    print("SMOKE 模式只预测了 64 条测试，不生成提交文件。")
else:
    submission = sample_frame[["id"]].copy()
    submission["sentiment"] = submission["id"].map(prediction)
    if submission["sentiment"].isna().any() or not submission["sentiment"].between(0, 1).all():
        raise ValueError("提交概率缺失或超出 [0, 1]")
    submission.to_csv(RUN_ROOT / "submission.csv", index=False)
    submission.assign(sentiment=(submission["sentiment"] >= 0.5).astype(int)).to_csv(
        RUN_ROOT / "test_labels.csv", index=False)
    SUMMARY["test_top1_is_answer"] = float(test_result["top_is_answer"].mean())
    SUMMARY["test_positive_rate"] = float((submission["sentiment"] >= 0.5).mean())
    print("提交文件:", RUN_ROOT / "submission.csv")

shutil.rmtree(RUN_ROOT / "checkpoints", ignore_errors=True)
save_json(RUN_ROOT / "summary.json", SUMMARY)
print(json.dumps({k: v for k, v in SUMMARY.items() if k not in ("config", "versions")}, indent=2, ensure_ascii=False))